# [재실행 전용 · 2026-10-04 회계 수정] crypto_hrl_earnhft_colab.ipynb

원본 `crypto_hrl_earnhft_colab.ipynb`에서 **이번 재실행에 필요한 셀만** 뽑은 노트북이다. 학습 셀(`.learn`)은 빠져 있고, 저장된 모델을 불러와 평가만 한다.
위에서부터 **순서대로 전부 실행**하면 된다. 결과 CSV는 원본과 같은 이름으로 `data/crypto/`에 저장된다(실행 전 백업 셀 먼저).

결과: 표 1 전체 행, 4.6절 β 단독 표(§16), 6.9절 거래비용 스윕(§18).


In [ ]:
import shutil, glob, os
from google.colab import drive
drive.mount('/content/drive')
src = '/content/drive/MyDrive/졸업프로젝트/data/crypto'
dst = src + '/_pre_accounting_fix_20261004'
if not os.path.exists(dst):
    os.makedirs(dst)
    for p in glob.glob(src + '/*.csv') + glob.glob(src + '/*.png'):
        shutil.copy2(p, dst)
print('백업 폴더:', dst, len(os.listdir(dst)), '개')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'
POOL_DIR   = MODEL_DIR / 'crypto_hrl_pool'

RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'
ORIG_PPO_PATH  = MODEL_DIR / 'crypto_ppo_portfolio.zip'   # crypto_rl_trading_colab.ipynb 산출물(있으면 비교에 포함)

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준)
BUCKET_SEC = 1800     # 30분 버킷

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'   # 열린 구간

# --- Stage A/B/C 하이퍼파라미터 ---
N_LABELS              = 5          # 레짐 라벨 수 (bear/pullback/sideways/rally/bull)
RISK_THRESHOLD        = 0.2        # EarnHFT θ — 상하위 라벨 경계 분위수 폭
LOWPASS_WINDOW        = 48         # 레짐 분할 저역통과 윈도우 = 1일(30분×48)
POOL_BETAS            = [-90, -30, 30, 90]   # Stage B 선호도 스윕 (논문 [-90,-10,30,100] 근사)
POOL_TIMESTEPS_EACH   = 100_000    # 풀 에이전트 1개당 학습 스텝 (원본 300k와 총량 비슷하게)
ROUTE_BUCKETS         = 48         # Stage C 라우팅 단위 = 1일
MACRO_WINDOW           = 96        # 라우터 상태에 포함할 최근 버킷 수 (=2일)
ROUTER_TOTAL_TIMESTEPS = 20_000    # 라우터(=일 단위) 학습 스텝

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음 — crypto_hybrid_model_colab.ipynb 먼저 실행'
print('경로 확인 완료')


## 1. RL 임베딩 로드 & 인덱스 구축

In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

# (symbol, 30분 버킷) → 임베딩 인덱스
emb_index  = {}
sym_ts_set = {}   # 심볼별로 임베딩 추출에 실제 쓰인 초 단위 timestamp 집합 (수익률 정확 매칭용)
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


## 2. 30분 수익률(RetTarget_6b) 로드

In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9   # ts_list와 동일 단위(초)
    y     = df['RetTarget_6b'].values
    # 임베딩 추출에 실제 쓰인 timestamp만 매칭 — 같은 버킷 내 다른 5분봉과 섞이지 않도록
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


## 3. CryptoPortfolioEnv (Gymnasium) — 기존과 동일, 변경 없음

In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        """종목별 임베딩 — 해당 버킷에 데이터 없으면 0-vector(상장 전/상폐 등)"""
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)  # (N*64,)

    def _asset_ret(self, bucket):
        """종목별 30분 수익률 — 없으면 0(그 자산은 보유해도 손익 없음으로 취급)"""
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        # Softmax로 비중 정규화
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        # 거래비용
        turnover = float(np.abs(action - self.weights).sum())  # 비중 변화량(진단용)
        tc_cost  = self.tc * turnover

        # RetTarget_6b는 현재 버킷 자체에 forward 수익률이 이미 인코딩되어 있음
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]
print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'테스트 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

test_env = CryptoPortfolioEnv(test_buckets, **env_kwargs)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')


## 4. Stage A — 시장 레짐 분할 (EarnHFT Algorithm 3)

In [ ]:
# crypto/market_segmentation.py 이식 (Colab은 저장소를 마운트하지 않으므로 동일 코드를 셀에 포함)
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    """가격 시계열을 레짐 구간으로 분할하고 라벨을 부여한다 (EarnHFT Algorithm 3 이식).
    로그가격(buy&hold net curve의 로그) 기준으로 계산해 가격 스케일 왜곡을 방지한다."""
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    """EarnHFT Eq.4 이식 — chunk 기울기(r)에 대한 β-지수가중 샘플링 우선순위(합 1)."""
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


In [ ]:
# 44종목 등가중 NAV 곡선 구성 (거래비용 제외 — 시장 자체의 추세 파악용)
eq_ret_per_bucket = np.array([
    np.mean([ret_index.get((sym, b), 0.0) for sym in symbols])
    for b in train_buckets
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_bucket), index=range(len(train_buckets)))

seg = segment_and_label(
    eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
    lowpass_window=LOWPASS_WINDOW,
)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.0f}버킷 ≈ {seg.chunks['length'].mean()/48:.1f}일)")
print(seg.chunks['regime'].value_counts())

fig, ax = plt.subplots(figsize=(14, 4))
colors = {'bear': 'firebrick', 'pullback': 'lightsalmon', 'sideways': 'lightgray', 'rally': 'lightgreen', 'bull': 'darkgreen'}
for _, row in seg.chunks.iterrows():
    ax.axvspan(row['start_idx'], row['end_idx'], color=colors[row['regime']], alpha=0.5)
ax.plot(eq_nav.values, color='black', lw=1)
ax.set_title('44종목 등가중 NAV + Stage A 레짐 분할 (2025 학습 구간)')
ax.set_xlabel('버킷 인덱스 (train_buckets 내 위치)')
plt.tight_layout()
plt.show()


## 5. Stage B — 레짐 선호도 기반 에이전트 풀 학습

In [ ]:
class ChunkedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    EarnHFT Stage II 이식 — reset()마다 레짐 우선순위(β)로 청크를 샘플링해
    그 구간만 진행하고 종료하는 에피소드형 환경.
    """

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], **env_kwargs)  # reset()에서 실제 구간으로 교체됨

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


# 스모크 테스트: β별로 몇 번 reset해서 실제로 다른 구간이 뽑히는지 확인
_test_env = ChunkedCryptoPortfolioEnv(
    all_buckets=train_buckets, chunks=seg.chunks, beta=90, rng=np.random.default_rng(0), **env_kwargs
)
for _ in range(3):
    _test_env.reset()
    print(f'β=90 샘플 구간: {len(_test_env.buckets)}버킷 ({bucket_to_date(_test_env.buckets[0])} ~ {bucket_to_date(_test_env.buckets[-1])})')


## 6. Stage C — 라우터 학습 (일 단위 풀 선택)

In [ ]:
class MacroRouterEnv(gym.Env):
    """
    EarnHFT Stage III 이식 — 1일 단위로 풀 중 하나를 선택해 그 정책이 해당 구간의
    실제 트레이딩을 대신 수행하도록 하는 고수준 환경.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, pool_models, route_buckets=48, macro_window=96, **env_kwargs):
        super().__init__()
        self.pool_models    = list(pool_models)
        self.route_buckets  = route_buckets
        self.macro_window   = macro_window
        self.inner           = CryptoPortfolioEnv(buckets=buckets, **env_kwargs)
        self.action_space      = spaces.Discrete(len(self.pool_models))
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(macro_window + 4,), dtype=np.float32)
        self._nav_history = []

    def _macro_obs(self):
        hist = self._nav_history[-self.macro_window:]
        if len(hist) < self.macro_window:
            hist = [1.0] * (self.macro_window - len(hist)) + hist
        recent = np.array(hist, dtype=np.float32)
        ret = np.diff(recent) if len(recent) > 1 else np.zeros(1, dtype=np.float32)
        stats = np.array([
            recent[-1] - recent[0],
            float(ret.std()),
            float(ret.mean()),
            self.inner.t / max(len(self.inner.buckets) - 1, 1),
        ], dtype=np.float32)
        return np.concatenate([recent, stats]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.inner.reset(seed=seed)
        self._nav_history = []
        return self._macro_obs(), {}

    def step(self, action):
        model = self.pool_models[int(action)]
        block_reward = 0.0
        done_inner = False
        for _ in range(self.route_buckets):
            obs = self.inner._get_obs()
            a, _ = model.predict(obs, deterministic=True)
            _, r, done_inner, _, info = self.inner.step(a)
            block_reward += r
            self._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        done = done_inner or self.inner.t >= len(self.inner.buckets) - 1
        info = {'portfolio_value': self.inner.portfolio_value}
        return self._macro_obs(), block_reward, done, False, info


# 스모크 테스트 — pool_models가 아직 없는 세션(§5를 건너뛰고 §5-로더만 나중에 실행하는 복구 경로)에서도
# 이 셀이 클래스 정의만은 항상 성공하도록 방어.
if 'pool_models' in globals() and pool_models:
    _router_probe = MacroRouterEnv(
        buckets=train_buckets[:500], pool_models=list(pool_models.values()),
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
    )
    obs, _ = _router_probe.reset()
    print('router obs shape:', obs.shape)
    obs, r, done, _, info = _router_probe.step(0)
    print('1스텝 후 reward:', r, 'portfolio_value:', info['portfolio_value'])
else:
    print('MacroRouterEnv 클래스 정의 완료 (pool_models 없음 — 스모크 테스트 생략, §5-로더 실행 후 계속 진행)')


In [ ]:
# 진단 재실행용 로더 — 런타임이 끊겨 pool_models/router가 메모리에 없어도
# 재학습 없이 저장된 모델만 불러와 백테스트를 이어서 할 수 있게 함.
# (풀 학습을 방금 마친 세션이면 이미 메모리에 있으므로 그대로 재사용됨)
if 'pool_models' not in globals() or not pool_models:
    pool_models = {b: PPO.load(str(POOL_DIR / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
    print('풀 에이전트 로드 완료:', list(pool_models.keys()))
if 'router' not in globals():
    router = DQN.load(str(MODEL_DIR / 'crypto_hrl_router.zip'))
    print('라우터 로드 완료')


## 7. 최종 백테스트 — 라우터+풀 vs 단일 PPO vs 등가중

In [ ]:
def compute_perf(values, steps_per_year=48 * 365):
    """성과 지표 계산 — 30분봉·24/7 시장 기준 연환산(48스텝/일)"""
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {
        '총 수익률':    f'{total_ret*100:.2f}%',
        '연환산 수익률': f'{annual_ret*100:.2f}%',
        'Sharpe':      f'{sharpe:.3f}',
        'MDD':         f'{max_dd*100:.2f}%',
    }


def turnover_stats(turnovers, tc=TC):
    """진단용 — 스텝별 비중 변화량(턴오버)과 거래비용 드래그 요약."""
    t = np.asarray(turnovers, dtype=np.float64)
    cost_factors = 1 - tc * t
    cost_drag = 1 - np.prod(cost_factors)  # 거래비용만으로 누적 손실된 비율
    return {
        '평균 턴오버/스텝':   f'{t.mean()*100:.1f}%',
        '중앙값 턴오버/스텝':  f'{np.median(t)*100:.1f}%',
        '스텝당 평균 거래비용': f'{(tc * t).mean()*100:.4f}%',
        '누적 거래비용 드래그(총자산 대비)': f'{cost_drag*100:.2f}%',
    }


def run_backtest(env, model, deterministic=True):
    obs, _ = env.reset()
    done, values = False, [1.0]
    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


def equal_weight_backtest(env):
    obs, _ = env.reset()
    done, values = False, [1.0]
    action = np.ones(env.n_assets, dtype=np.float32) / env.n_assets
    while not done:
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


print('테스트 구간에서 라우터+풀 시스템 실행 중...')
router_test_env = MacroRouterEnv(
    buckets=test_buckets, pool_models=[pool_models[b] for b in POOL_BETAS],
    route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
)
obs, _ = router_test_env.reset()
done, chosen = False, []
while not done:
    action, _ = router.predict(obs, deterministic=True)
    chosen.append(int(action))
    obs, _, done, _, info = router_test_env.step(action)

hrl_values = np.array(router_test_env.inner.value_history)
hrl_turnovers = list(router_test_env.inner.turnover_history)
print('라우터 선택 분포(β):', {POOL_BETAS[k]: v for k, v in pd.Series(chosen).value_counts().to_dict().items()})

comparison = {}
turnover_comparison = {}
if ORIG_PPO_PATH.exists():
    print('기존 단일 PPO(crypto_ppo_portfolio.zip) 로드해 비교에 포함...')
    orig_ppo = PPO.load(str(ORIG_PPO_PATH))
    ppo_values, ppo_turnovers = run_backtest(CryptoPortfolioEnv(test_buckets, **env_kwargs), orig_ppo)
    comparison['단일 PPO (기존)'] = compute_perf(ppo_values)
    turnover_comparison['단일 PPO (기존)'] = turnover_stats(ppo_turnovers)
else:
    print('crypto_ppo_portfolio.zip 없음 — 단일 PPO 비교 생략')

ew_values, ew_turnovers = equal_weight_backtest(CryptoPortfolioEnv(test_buckets, **env_kwargs))
comparison['등가중 벤치마크']    = compute_perf(ew_values)
turnover_comparison['등가중 벤치마크'] = turnover_stats(ew_turnovers)

comparison['EarnHFT식 라우터+풀'] = compute_perf(hrl_values)
turnover_comparison['EarnHFT식 라우터+풀'] = turnover_stats(hrl_turnovers)

perf_df = pd.DataFrame(comparison).T
print()
print(perf_df.to_string())
perf_df.to_csv(CRYPTO_DIR / 'crypto_hrl_performance.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_performance.csv')

turnover_df = pd.DataFrame(turnover_comparison).T
print()
print('--- 턴오버/거래비용 진단 ---')
print(turnover_df.to_string())
turnover_df.to_csv(CRYPTO_DIR / 'crypto_hrl_turnover_diag.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_turnover_diag.csv')


## 9. 스무딩 재백테스트 — 재학습 없이 관성 스무딩(α)만 적용

In [ ]:
%%time
# 관성 스무딩 스윕 — 학습된 모델은 그대로 두고 실행 단계의 비중 갱신만
#   w_new = (1-α)·w_old + α·softmax(action)
# 으로 바꾼다. softmax(log w) = w 이므로 log(w_new)를 action으로 넘기면
# env 내부 softmax가 w_new를 정확히 복원한다(환경 코드 수정 불필요).
# α=1.0은 스무딩 없음 → §7과 동일 조건의 기준선.
# 1차 스윕(0.3/0.15/0.05)에서 개선이 α=0.05까지 단조증가 → 0.03/0.02/0.01 추가해 최적점 탐색.

SMOOTH_ALPHAS = [1.0, 0.3, 0.15, 0.05, 0.03, 0.02, 0.01]


def _softmax_np(a):
    e = np.exp(a - a.max())
    return e / e.sum()


def run_hrl_smoothed(alpha):
    """라우터+풀 시스템을 테스트 구간에서 실행하되 저수준 비중에 관성 스무딩 적용.
    반환: (비용 후 NAV, 비용 전 NAV, 턴오버 리스트)"""
    env = MacroRouterEnv(
        buckets=test_buckets, pool_models=[pool_models[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
    )
    obs, _ = env.reset()
    inner = env.inner
    gross, g = [1.0], 1.0
    done = False
    while not done:
        r_action, _ = router.predict(obs, deterministic=True)
        model = env.pool_models[int(r_action)]
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            w = (1 - alpha) * inner.weights + alpha * _softmax_np(a)
            _, _, done_inner, _, info = inner.step(np.log(w))
            env._nav_history.append(info['portfolio_value'])
            g *= 1 + info['port_ret']
            gross.append(g)
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return np.array(inner.value_history), np.array(gross), list(inner.turnover_history)


def smooth_perf_row(net, gross, turnovers, steps_per_year=48 * 365):
    rets   = np.diff(net) / net[:-1]
    ann    = net[-1] ** (steps_per_year / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    mdd    = ((net / np.maximum.accumulate(net)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    return {
        '총 수익률(비용 후)': f'{(net[-1]-1)*100:.2f}%',
        '총 수익률(비용 전)': f'{(gross[-1]-1)*100:.2f}%',
        '연환산':           f'{ann*100:.2f}%',
        'Sharpe':          f'{sharpe:.3f}',
        'MDD':             f'{mdd*100:.2f}%',
        '평균 턴오버/스텝':    f'{t.mean()*100:.2f}%',
        '누적 비용 드래그':    f'{(1 - np.prod(1 - TC*t))*100:.2f}%',
    }


smooth_results, smooth_curves = {}, {}

# 참조용 등가중 (모델 predict 없이 즉시 계산)
_ew_env = CryptoPortfolioEnv(test_buckets, **env_kwargs)
_ew_env.reset()
_ew_a = np.ones(_ew_env.n_assets, dtype=np.float32) / _ew_env.n_assets
_ew_gross, _g, done = [1.0], 1.0, False
while not done:
    _, _, done, _, info = _ew_env.step(_ew_a)
    _g *= 1 + info['port_ret']
    _ew_gross.append(_g)
smooth_results['등가중 벤치마크'] = smooth_perf_row(
    np.array(_ew_env.value_history), np.array(_ew_gross), _ew_env.turnover_history)
smooth_curves['등가중 벤치마크'] = np.array(_ew_env.value_history)

_net_by_alpha = {}
for alpha in SMOOTH_ALPHAS:
    label = 'HRL α=1.0 (스무딩 없음)' if alpha == 1.0 else f'HRL α={alpha}'
    print(f'{label} 실행 중...')
    net, gross, turns = run_hrl_smoothed(alpha)
    smooth_results[label] = smooth_perf_row(net, gross, turns)
    smooth_curves[label]  = net
    _net_by_alpha[alpha]  = net[-1]

BEST_ALPHA = max((a for a in _net_by_alpha if a < 1.0), key=lambda a: _net_by_alpha[a])

smooth_df = pd.DataFrame(smooth_results).T
print()
print(smooth_df.to_string())
print(f'\n비용 후 최고 α = {BEST_ALPHA} (총 수익률 {(_net_by_alpha[BEST_ALPHA]-1)*100:.2f}%) → §10 재학습에 사용')
smooth_df.to_csv(CRYPTO_DIR / 'crypto_hrl_smoothing_sweep.csv', encoding='utf-8-sig')
print('저장 완료: crypto_hrl_smoothing_sweep.csv')

fig, ax = plt.subplots(figsize=(14, 5))
for label, curve in smooth_curves.items():
    lw, ls = (1.5, '--') if '등가중' in label else (1.8, '-')
    ax.plot(curve, label=label, lw=lw, ls=ls)
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_title(f'관성 스무딩 α별 누적 가치 (테스트: {RL_TEST_START} ~, 비용 후)', fontsize=13)
ax.set_ylabel('포트폴리오 가치 (초기=1.0)')
ax.set_xlabel('30분 스텝')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hrl_smoothing_sweep.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. v2 재학습 — 스무딩을 학습 환경에 내장 + 라우터에 시장 피처 추가

In [ ]:
# §10-A — 스무딩 내장 환경(v2) + 시장 피처 라우터 정의
SMOOTH_ALPHA_TRAIN = 0.05
if 'BEST_ALPHA' in globals():
    # α가 너무 작으면 행동→보상 신호가 약해져 PPO 학습이 어려우므로 하한 0.03
    SMOOTH_ALPHA_TRAIN = max(BEST_ALPHA, 0.03)
print(f'학습용 스무딩 계수 α = {SMOOTH_ALPHA_TRAIN}')


class SmoothStepMixin:
    """step()에서 w_new = (1-α)·w_old + α·softmax(action)을 적용한 뒤 log(w_new)를
    부모 step에 넘긴다 — 부모의 softmax가 w_new를 그대로 복원하므로 부모 코드 수정 불필요."""
    smooth_alpha = 0.05

    def step(self, action):
        target = np.exp(action - np.max(action))
        target = target / target.sum()
        w = (1 - self.smooth_alpha) * self.weights + self.smooth_alpha * target
        return super().step(np.log(w))


class SmoothedPortfolioEnv(SmoothStepMixin, CryptoPortfolioEnv):
    def __init__(self, *args, smooth_alpha=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.smooth_alpha = smooth_alpha


class SmoothedChunkedEnv(SmoothStepMixin, ChunkedCryptoPortfolioEnv):
    def __init__(self, *args, smooth_alpha=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.smooth_alpha = smooth_alpha


# 라우터 관측용 등가중 시장 수익률. RetTarget_6b는 forward 수익률(버킷 b 값 = b→b+1 구간)이므로
# 시점 t의 관측에는 t-1 이하 버킷 값만 넣는다 — look-ahead 차단은 _macro_obs의 [:t] 슬라이스가 담당.
ew_ret_by_bucket = {
    b: float(np.mean([ret_index.get((s, b), 0.0) for s in symbols]))
    for b in all_buckets
}


class MacroRouterEnvV2(MacroRouterEnv):
    """v2 — (1) 내부 환경에 관성 스무딩 내장, (2) 관측에 등가중 시장 수익률 창 추가."""

    def __init__(self, buckets, pool_models, ew_ret_by_bucket, smooth_alpha=0.05,
                 route_buckets=48, macro_window=96, **env_kwargs):
        super().__init__(buckets, pool_models, route_buckets=route_buckets,
                         macro_window=macro_window, **env_kwargs)
        self.inner  = SmoothedPortfolioEnv(buckets=buckets, smooth_alpha=smooth_alpha, **env_kwargs)
        self.ew_ret = ew_ret_by_bucket
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(macro_window * 2 + 8,), dtype=np.float32)

    def _macro_obs(self):
        base = super()._macro_obs()   # 자기 NAV 창 + 4개 통계
        t = self.inner.t
        win = np.zeros(self.macro_window, dtype=np.float32)
        past = [self.ew_ret.get(b, 0.0) for b in self.inner.buckets[max(0, t - self.macro_window):t]]
        if past:
            win[-len(past):] = past
        win *= 100.0   # 30분 수익률(~1e-3) 스케일 보정
        day = win[-self.route_buckets:]
        stats = np.array([win.mean(), win.std(), day.mean(), day.std()], dtype=np.float32)
        return np.concatenate([base, win, stats]).astype(np.float32)


# 스모크 테스트 — v1 풀로 관측 shape와 스무딩 적용 여부 확인
_v2_probe = MacroRouterEnvV2(
    buckets=train_buckets[:500], pool_models=[pool_models[b] for b in POOL_BETAS],
    ew_ret_by_bucket=ew_ret_by_bucket, smooth_alpha=SMOOTH_ALPHA_TRAIN,
    route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
)
obs, _ = _v2_probe.reset()
print('v2 router obs shape:', obs.shape)
obs, r, done, _, info = _v2_probe.step(0)
print(f'1스텝(=1일) 후 reward: {r:.5f} | 내부 턴오버(스무딩 적용): '
      f'{np.mean(_v2_probe.inner.turnover_history)*100:.2f}%/스텝')


In [ ]:
# §10 진단 재실행용 로더 — 세션이 끊겨도 저장된 v2 모델만 다시 불러오면 백테스트 가능
# (§10-A의 클래스 정의 셀은 먼저 실행돼 있어야 함)
if 'pool_v2' not in globals() or not pool_v2:
    pool_v2 = {b: PPO.load(str(POOL_DIR / f'agent_v2_beta_{b}.zip')) for b in POOL_BETAS}
    print('[v2] 풀 로드 완료:', list(pool_v2.keys()))
if 'router_v2' not in globals():
    router_v2 = DQN.load(str(MODEL_DIR / 'crypto_hrl_router_v2.zip'))
    print('[v2] 라우터 로드 완료')


In [ ]:
# §10-D — 최종 비교: v2(스무딩 내장 재학습) vs v1+추론 스무딩 vs 등가중
def _perf_v2(net, gross, turnovers, steps_per_year=48 * 365):
    rets   = np.diff(net) / net[:-1]
    ann    = net[-1] ** (steps_per_year / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    mdd    = ((net / np.maximum.accumulate(net)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    return {
        '총 수익률(비용 후)': f'{(net[-1]-1)*100:.2f}%',
        '총 수익률(비용 전)': f'{(gross[-1]-1)*100:.2f}%',
        '연환산':           f'{ann*100:.2f}%',
        'Sharpe':          f'{sharpe:.3f}',
        'MDD':             f'{mdd*100:.2f}%',
        '평균 턴오버/스텝':    f'{t.mean()*100:.2f}%',
        '누적 비용 드래그':    f'{(1 - np.prod(1 - TC*t))*100:.2f}%',
    }


print('[v2] 테스트 구간 실행 중...')
v2_env = MacroRouterEnvV2(
    buckets=test_buckets, pool_models=[pool_v2[b] for b in POOL_BETAS],
    ew_ret_by_bucket=ew_ret_by_bucket, smooth_alpha=SMOOTH_ALPHA_TRAIN,
    route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
)
obs, _ = v2_env.reset()
done, chosen_v2, v2_gross, g = False, [], [1.0], 1.0
while not done:
    a, _ = router_v2.predict(obs, deterministic=True)
    chosen_v2.append(int(a))
    # env.step과 동일한 저수준 루프 — gross(비용 전) NAV 추적을 위해 수동 진행
    model = v2_env.pool_models[int(a)]
    inner = v2_env.inner
    done_inner = False
    for _ in range(v2_env.route_buckets):
        pa, _ = model.predict(inner._get_obs(), deterministic=True)
        _, _, done_inner, _, info = inner.step(pa)
        v2_env._nav_history.append(info['portfolio_value'])
        g *= 1 + info['port_ret']
        v2_gross.append(g)
        if done_inner:
            break
    obs = v2_env._macro_obs()
    done = done_inner or inner.t >= len(inner.buckets) - 1

v2_net = np.array(v2_env.inner.value_history)
print('[v2] 라우터 선택 분포(β):',
      {POOL_BETAS[k]: v for k, v in pd.Series(chosen_v2).value_counts().to_dict().items()})

final_results, final_curves = {}, {}

# 등가중 벤치마크
_ew = CryptoPortfolioEnv(test_buckets, **env_kwargs)
_ew.reset()
_ewa = np.ones(_ew.n_assets, dtype=np.float32) / _ew.n_assets
_ewg, _gg, d = [1.0], 1.0, False
while not d:
    _, _, d, _, info = _ew.step(_ewa)
    _gg *= 1 + info['port_ret']
    _ewg.append(_gg)
final_results['등가중 벤치마크'] = _perf_v2(np.array(_ew.value_history), np.array(_ewg), _ew.turnover_history)
final_curves['등가중 벤치마크'] = np.array(_ew.value_history)

# v1 + 추론 스무딩 (§9의 run_hrl_smoothed와 v1 모델이 세션에 있으면 포함)
if 'run_hrl_smoothed' in globals() and 'pool_models' in globals() and 'router' in globals():
    _l1 = f'v1 + 추론 스무딩 α={SMOOTH_ALPHA_TRAIN}'
    print(f'{_l1} 실행 중...')
    net1, gross1, turns1 = run_hrl_smoothed(SMOOTH_ALPHA_TRAIN)
    final_results[_l1] = _perf_v2(net1, gross1, turns1)
    final_curves[_l1]  = net1

_l2 = f'v2 재학습 (α={SMOOTH_ALPHA_TRAIN} 내장)'
final_results[_l2] = _perf_v2(v2_net, np.array(v2_gross), v2_env.inner.turnover_history)
final_curves[_l2]  = v2_net

final_df = pd.DataFrame(final_results).T
print()
print(final_df.to_string())
final_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v2_performance.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v2_performance.csv')

fig, ax = plt.subplots(figsize=(14, 5))
for label, curve in final_curves.items():
    lw, ls = (1.5, '--') if '등가중' in label else (1.8, '-')
    ax.plot(curve, label=label, lw=lw, ls=ls)
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_title(f'v2 재학습 vs v1 추론 스무딩 vs 등가중 (테스트: {RL_TEST_START} ~, 비용 후)', fontsize=13)
ax.set_ylabel('포트폴리오 가치 (초기=1.0)')
ax.set_xlabel('30분 스텝')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hrl_v2_backtest.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. v3 — EarnHFT 원문 충실 재구현 (현금 액션 + Stage II 선별 + Stage III 검증 선택)

In [ ]:
# §12-A — 현금 액션 환경 + 2025 내부 train/valid 분할 + v3 라우터 환경 정의
V3_TRAIN_END   = '2025-09-30'   # Stage B 학습: 2025-01 ~ 2025-09
V3_VALID_START = '2025-10-01'   # 선별/검증: 2025-10 ~ 2025-12 (테스트 2026은 최종 1회만)
V3_POOL_DIR    = MODEL_DIR / 'crypto_hrl_pool_v3'
V3_INFER_ALPHA = 0.03           # 선별 단계 실행 스무딩 기본값 — 최종 α는 §12-F에서 valid로 선택

v3_train_buckets = [b for b in train_buckets if bucket_to_date(b) <= V3_TRAIN_END]
v3_valid_buckets = [b for b in train_buckets if bucket_to_date(b) >= V3_VALID_START]
print(f'v3 학습: {bucket_to_date(v3_train_buckets[0])} ~ {bucket_to_date(v3_train_buckets[-1])} ({len(v3_train_buckets):,}스텝)')
print(f'v3 검증: {bucket_to_date(v3_valid_buckets[0])} ~ {bucket_to_date(v3_valid_buckets[-1])} ({len(v3_valid_buckets):,}스텝)')

if 'ew_ret_by_bucket' not in globals():
    ew_ret_by_bucket = {
        b: float(np.mean([ret_index.get((s, b), 0.0) for s in symbols]))
        for b in all_buckets
    }


class CashPortfolioEnv(CryptoPortfolioEnv):
    """44종목 + 현금(마지막 인덱스, 수익률 0) — 논문의 'position 0'(청산) 대응.
    β<0 에이전트가 하락장 대응(포지션 축소)을 액션으로 직접 표현할 수 있게 한다.
    거래비용은 자산 레그 턴오버에만 부과(현금 이동 자체는 무료)."""

    def __init__(self, buckets, **kwargs):
        super().__init__(buckets, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(self.n_assets * self.emb_dim + self.n_assets + 1,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        gym.Env.reset(self, seed=seed)
        self.t                = 0
        self.weights          = _mask_untradable(np.ones(self.n_assets + 1, dtype=np.float32),
                                          np.append(self._tradable(self.buckets[0]), True))  # 거래 가능 종목+현금 등가중에서 시작
        self.portfolio_value  = 1.0
        self.value_history    = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        w = np.exp(action - action.max())
        w = w / w.sum()                                                   # 45-dim, 합=1
        w = _mask_untradable(w, np.append(self._tradable(self.buckets[self.t]), True))  # 거래 불가 종목 제외
        asset_turnover = float(np.abs(w[:-1] - self.weights[:-1]).sum())  # 자산 레그만
        tc_cost = self.tc * asset_turnover

        b        = self.buckets[self.t]
        port_ret = float((w[:-1] * self._asset_ret(b)).sum())             # 현금 수익률 0

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights = _drift_weights(w, np.append(self._asset_ret(b), 0.0))  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(asset_turnover)
        self.t += 1

        done = (self.t >= len(self.buckets) - 1)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': asset_turnover, 'tc_cost': tc_cost, 'cash': float(w[-1])}
        return self._get_obs(), float(port_ret - tc_cost), done, False, info


class ChunkedCashEnv(CashPortfolioEnv):
    """β 우선순위 청크 샘플링 에피소드 — §5 ChunkedCryptoPortfolioEnv와 동일 로직, 환경만 현금 버전."""

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], **env_kwargs)

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


class _V3SmoothMixin:
    """실행 스무딩 — 학습이 아닌 실행(라우터 내부·평가)에서만 사용 (v2 실패 교훈)."""
    smooth_alpha = V3_INFER_ALPHA

    def step(self, action):
        target = np.exp(action - np.max(action))
        target = target / target.sum()
        w = (1 - self.smooth_alpha) * self.weights + self.smooth_alpha * target
        return super().step(np.log(w))


class SmoothedCashEnv(_V3SmoothMixin, CashPortfolioEnv):
    def __init__(self, *args, smooth_alpha=V3_INFER_ALPHA, **kwargs):
        super().__init__(*args, **kwargs)
        self.smooth_alpha = smooth_alpha




    # CashRouterEnv는 §6(MacroRouterEnv)이 정의돼 있어야 만들 수 있다 — §21(다양성/목적함수 주입
# 검증)은 라우터를 쓰지 않으므로 §6 없이도 §12-A가 끝까지 돌아가도록 조건부로만 정의한다.
if 'MacroRouterEnv' in globals():
    class CashRouterEnv(MacroRouterEnv):
        """§6 라우터 환경에 (1) 현금 내부 환경 + 실행 스무딩, (2) 시장 수익률 관측(논문 Xht 대응) 추가."""

        def __init__(self, buckets, pool_models, ew_ret_by_bucket, smooth_alpha=V3_INFER_ALPHA,
                     route_buckets=48, macro_window=96, **env_kwargs):
            super().__init__(buckets, pool_models, route_buckets=route_buckets,
                             macro_window=macro_window, **env_kwargs)
            self.inner  = SmoothedCashEnv(buckets=buckets, smooth_alpha=smooth_alpha, **env_kwargs)
            self.ew_ret = ew_ret_by_bucket
            self.observation_space = spaces.Box(
                -np.inf, np.inf, shape=(macro_window * 2 + 8,), dtype=np.float32)

        def _macro_obs(self):
            base = super()._macro_obs()   # 자기 NAV 창 + 4개 통계
            t = self.inner.t
            win = np.zeros(self.macro_window, dtype=np.float32)
            past = [self.ew_ret.get(b, 0.0) for b in self.inner.buckets[max(0, t - self.macro_window):t]]
            if past:
                win[-len(past):] = past
            win *= 100.0   # 30분 수익률(~1e-3) 스케일 보정
            day = win[-self.route_buckets:]
            stats = np.array([win.mean(), win.std(), day.mean(), day.std()], dtype=np.float32)
            return np.concatenate([base, win, stats]).astype(np.float32)
else:
    print('[§12-A] MacroRouterEnv 없음(§6 미실행) — CashRouterEnv 정의 생략(§21은 이 클래스 필요 없음)')



check_env(CashPortfolioEnv(v3_train_buckets[:100], **env_kwargs), warn=True)
print('CashPortfolioEnv 유효성 OK (obs dim =', 44 * EMB_DIM + 45, ')')


In [ ]:
# §12-B — Stage A 재실행: v3 학습구간 청크(β 샘플링용) + 검증구간 라벨(에이전트 선별용)
def ew_nav_of(buckets):
    r = np.array([ew_ret_by_bucket.get(b, 0.0) for b in buckets])
    return pd.Series(np.cumprod(1 + r), index=range(len(buckets)))

seg_v3_train = segment_and_label(
    ew_nav_of(v3_train_buckets), n_labels=N_LABELS,
    risk_threshold=RISK_THRESHOLD, lowpass_window=LOWPASS_WINDOW,
)
seg_v3_valid = segment_and_label(
    ew_nav_of(v3_valid_buckets), n_labels=N_LABELS,
    risk_threshold=RISK_THRESHOLD, lowpass_window=LOWPASS_WINDOW,
)
print(f"학습 청크: {len(seg_v3_train.chunks)}개 | 레짐 분포: {seg_v3_train.chunks['regime'].value_counts().to_dict()}")
print(f"검증 청크: {len(seg_v3_valid.chunks)}개 | 레짐 분포: {seg_v3_valid.chunks['regime'].value_counts().to_dict()}")


In [ ]:
# §12-E에서 함수 정의만 발췌 (라우터 학습 부분 제외)
def run_router_v3(rt, buckets, alpha=V3_INFER_ALPHA):
    """라우터+풀 실행 (수동 저수준 루프 — gross NAV·현금 비중 추적).
    반환: (비용 후 NAV, 비용 전 NAV, 턴오버, 현금비중 히스토리, 선택 로그)"""
    env = CashRouterEnv(
        buckets=buckets, pool_models=v3_pool_list, ew_ret_by_bucket=ew_ret_by_bucket,
        smooth_alpha=alpha, route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
    )
    obs, _ = env.reset()
    inner = env.inner
    gross, g, cash_hist, chosen, done = [1.0], 1.0, [], [], False
    while not done:
        a, _ = rt.predict(obs, deterministic=True)
        chosen.append(int(a))
        model = env.pool_models[int(a)]
        done_inner = False
        for _ in range(env.route_buckets):
            pa, _ = model.predict(inner._get_obs(), deterministic=True)
            _, _, done_inner, _, info = inner.step(pa)
            env._nav_history.append(info['portfolio_value'])
            g *= 1 + info['port_ret']
            gross.append(g)
            cash_hist.append(info['cash'])
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return (np.array(inner.value_history), np.array(gross),
            list(inner.turnover_history), cash_hist, chosen)




In [ ]:
# §12 재실행용 로더 — 세션이 끊겨도 §12-A·B 실행 후 이 셀로 풀/라우터 복구 가능
if 'v3_pool_list' not in globals():
    _sel = pd.read_csv(CRYPTO_DIR / 'crypto_hrl_v3_pool_selection.csv')
    v3_pool_list = [PPO.load(str(V3_POOL_DIR / f'v3_beta_{int(r.beta)}_ck{int(r.steps)}.zip'))
                    for r in _sel.itertuples()]
    print(f'[v3] 풀 로드: {len(v3_pool_list)}개 ({list(_sel.regime)})')
if 'router_v3_best' not in globals():
    router_v3_best = DQN.load(str(MODEL_DIR / 'crypto_hrl_router_v3.zip'))
    print('[v3] 라우터 로드 완료')


In [ ]:
%%time
# §12-F — 실행 α를 valid에서 선택(테스트 누수 제거) 후, 2026 테스트는 최종 구성으로 단 1회 실행
def _perf_v3(net, gross, turnovers, steps_per_year=48 * 365):
    rets   = np.diff(net) / net[:-1]
    ann    = net[-1] ** (steps_per_year / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    mdd    = ((net / np.maximum.accumulate(net)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    return {
        '총 수익률(비용 후)': f'{(net[-1]-1)*100:.2f}%',
        '총 수익률(비용 전)': f'{(gross[-1]-1)*100:.2f}%',
        '연환산':           f'{ann*100:.2f}%',
        'Sharpe':          f'{sharpe:.3f}',
        'MDD':             f'{mdd*100:.2f}%',
        '평균 턴오버/스텝':    f'{t.mean()*100:.2f}%',
    }


print('valid에서 실행 α 선택...')
alpha_valid = {}
for alpha in [0.05, 0.03, 0.02, 0.01]:
    vals, _, _, _, _ = run_router_v3(router_v3_best, v3_valid_buckets, alpha=alpha)
    alpha_valid[alpha] = vals[-1] - 1
    print(f'  α={alpha}: valid {(vals[-1]-1)*100:+.2f}%')
V3_FINAL_ALPHA = max(alpha_valid, key=alpha_valid.get)
print(f'▶ 최종 α = {V3_FINAL_ALPHA} (valid 기준 선택 — 테스트 데이터 미사용)\n')

print('=== 2026 테스트 최종 1회 실행 ===')
v3_net, v3_gross, v3_turns, v3_cash, v3_chosen = run_router_v3(
    router_v3_best, test_buckets, alpha=V3_FINAL_ALPHA)
_sel = pd.read_csv(CRYPTO_DIR / 'crypto_hrl_v3_pool_selection.csv')
_regimes = list(_sel.regime)
print('라우터 선택 분포:', {_regimes[k]: v for k, v in pd.Series(v3_chosen).value_counts().to_dict().items()})
print(f'평균 현금 비중: {np.mean(v3_cash)*100:.1f}% | 최대: {np.max(v3_cash)*100:.1f}%')

final_results, final_curves = {}, {}

_ew = CashPortfolioEnv(test_buckets, **env_kwargs)
_ew.reset()
_ewa = np.zeros(_ew.n_assets + 1, dtype=np.float32)
_ewa[:-1] = 1.0 / _ew.n_assets            # 등가중 44종목, 현금 0 — log-softmax 통과용
_ewg, _gg, d = [1.0], 1.0, False
_ew_logit = np.log(np.maximum(_ewa, 1e-12))
while not d:
    _, _, d, _, info = _ew.step(_ew_logit)
    _gg *= 1 + info['port_ret']
    _ewg.append(_gg)
final_results['등가중 벤치마크'] = _perf_v3(np.array(_ew.value_history), np.array(_ewg), _ew.turnover_history)
final_curves['등가중 벤치마크'] = np.array(_ew.value_history)

final_results[f'v3 (현금+선별+검증, α={V3_FINAL_ALPHA})'] = _perf_v3(v3_net, v3_gross, v3_turns)
final_curves[f'v3 (현금+선별+검증, α={V3_FINAL_ALPHA})'] = v3_net

final_df = pd.DataFrame(final_results).T
print()
print('참고(테스트 구간에서 α를 고른 §9 결과 — 직접 비교 시 주의): v1+스무딩 α=0.01 = +11.05% / Sharpe 0.638 / MDD -36.47%')
print(final_df.to_string())
final_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_performance.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v3_performance.csv')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[2, 1], sharex=True)
ax = axes[0]
for label, curve in final_curves.items():
    lw, ls = (1.5, '--') if '등가중' in label else (2.0, '-')
    ax.plot(curve, label=label, lw=lw, ls=ls)
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_title(f'v3 최종 백테스트 (테스트: {RL_TEST_START} ~, 비용 후)', fontsize=13)
ax.set_ylabel('포트폴리오 가치 (초기=1.0)')
ax.legend()
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.fill_between(range(len(v3_cash)), v3_cash, 0, alpha=0.4, color='gray')
ax2.set_title('v3 현금 비중 추이 (논문 position 0 대응 — 하락장에서 올라가야 정상)', fontsize=11)
ax2.set_ylabel('현금 비중')
ax2.set_xlabel('30분 스텝')
ax2.set_ylim(0, 1)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hrl_v3_backtest.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. v4 — FreQuant Top-K 희소선택 + DeepClair 리스크게이트 (저수준 액션만 교체)

### 13-A. SparseGatedCryptoPortfolioEnv 정의

In [ ]:
TOP_K = 10   # 종목 유니버스(44) 중 매 스텝 보유 가능한 최대 종목 수 (FreQuant top-|G| 이식)

class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    FreQuant(top-K 희소 선택) + DeepClair(전용 리스크게이트 ρ) 결합 저수준 환경.

    Action: (N+1,)-dim — 앞 N개는 종목별 confidence logit(FreQuant Eq.13-14의 h_i 역할),
            마지막 1개는 리스크자산 총 노출 게이트 로짓(DeepClair의 ρ_t 역할).
    변환:   top-K confidence logit만 골라 그 안에서 softmax(FreQuant Eq.14) →
            sigmoid(게이트)를 곱해 최종 종목별 비중. 나머지는 암묵적 현금(수익률 0).
    관측·NAV 회계는 CryptoPortfolioEnv와 동일. step_with_weights()는 목표 비중을 직접
    받아 NAV를 갱신하는 보조 메서드 — §13-G 관성 스무딩에서 (top-K 변환을 다시 태우지 않고)
    이미 블렌딩된 비중을 그대로 반영하기 위해 필요하다.
    """

    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        asset_logits = np.where(self._tradable(self.buckets[self.t]), asset_logits, -np.inf)  # 거래 불가 종목 제외
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        target_w = _mask_untradable(target_w, self._tradable(self.buckets[self.t]), renorm=False)  # 빠진 몫은 현금
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(target_w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


# 스모크 테스트
_sg_env = SparseGatedCryptoPortfolioEnv(test_buckets[:200], top_k=TOP_K, **env_kwargs)
check_env(_sg_env, warn=True)
obs, _ = _sg_env.reset()
a = np.random.randn(_sg_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _sg_env.step(a)
print(f"action space: {_sg_env.action_space.shape} | 선택된 종목 수: {len(info['selected'])} | "
      f"rho: {info['rho']:.3f} | reward: {r:.5f}")

### 13-B. Stage B용 청크 샘플링 환경 (§5의 β 우선순위 로직 재사용)

In [ ]:
class ChunkedSparseGatedEnv(SparseGatedCryptoPortfolioEnv):
    """§5의 ChunkedCryptoPortfolioEnv와 동일한 β-우선순위 청크 샘플링, 저수준 환경만 v4로 교체."""

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, top_k=TOP_K, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], top_k=top_k, **env_kwargs)  # reset()에서 실제 구간으로 교체됨

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


# 스모크 테스트
_test_env_v4 = ChunkedSparseGatedEnv(
    all_buckets=train_buckets, chunks=seg.chunks, beta=90, rng=np.random.default_rng(0), **env_kwargs
)
for _ in range(3):
    _test_env_v4.reset()
    print(f"β=90 샘플 구간: {len(_test_env_v4.buckets)}버킷 "
          f"({bucket_to_date(_test_env_v4.buckets[0])} ~ {bucket_to_date(_test_env_v4.buckets[-1])})")

### 13-D. Stage C — v4 라우터 (§6과 동일 구조·예산, 저수준 환경만 교체)

In [ ]:
class MacroRouterEnvV4(gym.Env):
    """§6 MacroRouterEnv와 동일 로직, 저수준 환경만 SparseGatedCryptoPortfolioEnv로 교체.

    episode_progress_mode: 관측값 마지막 성분(t/len(buckets), '에피소드 내 진행률').
      - 'real'(기본값, §13-C/§13-D 학습에 사용): 실제 진행률을 그대로 넘김 — 원 EarnHFT
        라우터 관측 설계를 그대로 재현하지만, 실거래에는 '에피소드 끝'이 없어
        policy_v4.py는 이 값을 상수 0.5로 근사한다(라이브가 학습 때 없던 입력 분포를
        받는 구조적 불일치, 모듈 docstring 한계 3).
      - 'constant': 학습 중에도 항상 0.5로 고정 — 라이브가 실제로 보게 될 입력 분포와
        학습 분포를 일치시킨 '매칭' 라우터(§13-I)용.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, pool_models, route_buckets=48, macro_window=96, top_k=TOP_K,
                 episode_progress_mode='real', **env_kwargs):
        super().__init__()
        self.pool_models           = list(pool_models)
        self.route_buckets         = route_buckets
        self.macro_window          = macro_window
        self.episode_progress_mode = episode_progress_mode
        self.inner           = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
        self.action_space      = spaces.Discrete(len(self.pool_models))
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(macro_window + 4,), dtype=np.float32)
        self._nav_history = []

    def _macro_obs(self):
        hist = self._nav_history[-self.macro_window:]
        if len(hist) < self.macro_window:
            hist = [1.0] * (self.macro_window - len(hist)) + hist
        recent = np.array(hist, dtype=np.float32)
        ret = np.diff(recent) if len(recent) > 1 else np.zeros(1, dtype=np.float32)
        if self.episode_progress_mode == 'constant':
            episode_progress = 0.5
        else:
            episode_progress = self.inner.t / max(len(self.inner.buckets) - 1, 1)
        stats = np.array([
            recent[-1] - recent[0],
            float(ret.std()),
            float(ret.mean()),
            episode_progress,
        ], dtype=np.float32)
        return np.concatenate([recent, stats]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.inner.reset(seed=seed)
        self._nav_history = []
        return self._macro_obs(), {}

    def step(self, action):
        model = self.pool_models[int(action)]
        block_reward = 0.0
        done_inner = False
        for _ in range(self.route_buckets):
            obs = self.inner._get_obs()
            a, _ = model.predict(obs, deterministic=True)
            _, r, done_inner, _, info = self.inner.step(a)
            block_reward += r
            self._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        done = done_inner or self.inner.t >= len(self.inner.buckets) - 1
        info = {'portfolio_value': self.inner.portfolio_value}
        return self._macro_obs(), block_reward, done, False, info


# 스모크 테스트 — pool_models_v4가 아직 없는 세션(§13-C를 건너뛰고 §13-로더만 나중에 실행하는
# 복구 경로)에서도 이 셀이 클래스 정의만은 항상 성공하도록 방어.
if 'pool_models_v4' in globals() and pool_models_v4:
    _router_probe_v4 = MacroRouterEnvV4(
        buckets=train_buckets[:500], pool_models=list(pool_models_v4.values()),
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
    )
    obs, _ = _router_probe_v4.reset()
    print('[v4] router obs shape:', obs.shape)
    obs, r, done, _, info = _router_probe_v4.step(0)
    print('[v4] 1스텝 후 reward:', r, 'portfolio_value:', info['portfolio_value'])
else:
    print('MacroRouterEnvV4 클래스 정의 완료 (pool_models_v4 없음 — 스모크 테스트 생략, §13-로더 실행 후 계속 진행)')


In [ ]:
# [v4] 진단 재실행용 로더 — 세션이 끊겨도 저장된 모델만 불러와 이어서 백테스트 가능
# (§13-C 학습 셀을 건너뛴 경로에서는 POOL_DIR_V4가 아직 없으므로 여기서 재정의 — 경로만 가리킬 뿐 idempotent)
POOL_DIR_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
if 'pool_models_v4' not in globals() or not pool_models_v4:
    pool_models_v4 = {b: PPO.load(str(POOL_DIR_V4 / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
    print('[v4] 풀 에이전트 로드 완료:', list(pool_models_v4.keys()))
if 'router_v4' not in globals():
    router_v4 = DQN.load(str(MODEL_DIR / 'crypto_hrl_router_v4.zip'))
    print('[v4] 라우터 로드 완료')

# 버그 수정: 이 로더만 실행하고 §13-B(셀 46)를 건너뛴 세션에서는 ChunkedSparseGatedEnv가
# 정의되지 않아 §13-G 다양성 진단(β별 단독 프로브)이 NameError로 죽는다 — 여기서 idempotent하게 복구.
if 'ChunkedSparseGatedEnv' not in globals():
    class ChunkedSparseGatedEnv(SparseGatedCryptoPortfolioEnv):
        """§13-B(셀 46)와 동일 정의 — 재실행 경로에서 누락됐을 때만 여기서 복구."""

        def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, top_k=TOP_K, **env_kwargs):
            self.all_buckets = all_buckets
            self.chunks      = chunks.reset_index(drop=True)
            self.beta        = beta
            self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
            self.rng         = rng or np.random.default_rng()
            super().__init__(buckets=all_buckets[:2], top_k=top_k, **env_kwargs)  # reset()에서 실제 구간으로 교체됨

        def reset(self, seed=None, options=None):
            idx = self.rng.choice(len(self.chunks), p=self.priority)
            row = self.chunks.iloc[idx]
            self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
            return super().reset(seed=seed, options=options)

    print('[v4] ChunkedSparseGatedEnv 재정의 완료 (§13-B 건너뛴 세션 대비 — §13-G 진단에 필요)')


### 13-E. 최종 백테스트 — v4 vs v1(+스무딩) vs 등가중

In [ ]:
# 방어적 재계산 — §7/§9를 건너뛰고 로더 셀만 실행한 세션 대비
if 'ew_values' not in globals():
    ew_values, ew_turnovers = equal_weight_backtest(CryptoPortfolioEnv(test_buckets, **env_kwargs))
if 'hrl_values' not in globals():
    _v1_test_env = MacroRouterEnv(
        buckets=test_buckets, pool_models=[pool_models[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, **env_kwargs,
    )
    _obs, _ = _v1_test_env.reset()
    _done = False
    while not _done:
        _a, _ = router.predict(_obs, deterministic=True)
        _obs, _, _done, _, _ = _v1_test_env.step(_a)
    hrl_values    = np.array(_v1_test_env.inner.value_history)
    hrl_turnovers = list(_v1_test_env.inner.turnover_history)

print('[v4] 테스트 구간에서 라우터+풀 시스템 실행 중...')
router_test_env_v4 = MacroRouterEnvV4(
    buckets=test_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
    route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
)
obs, _ = router_test_env_v4.reset()
done, chosen_v4 = False, []
while not done:
    action, _ = router_v4.predict(obs, deterministic=True)
    chosen_v4.append(int(action))
    obs, _, done, _, info = router_test_env_v4.step(action)

v4_values    = np.array(router_test_env_v4.inner.value_history)
v4_turnovers = list(router_test_env_v4.inner.turnover_history)
print('[v4] 라우터 선택 분포(β):', {POOL_BETAS[k]: v for k, v in pd.Series(chosen_v4).value_counts().to_dict().items()})

print(f'[v1 참조] 관성 스무딩 α={BEST_ALPHA} 재실행 중...')
v1_smoothed_net, _, v1_smoothed_turnovers = run_hrl_smoothed(BEST_ALPHA)

v4_comparison = {
    '등가중 벤치마크':                     compute_perf(ew_values),
    'v1 라우터+풀 (스무딩 없음)':            compute_perf(hrl_values),
    f'v1 + 스무딩 α={BEST_ALPHA}':          compute_perf(v1_smoothed_net),
    'v4 라우터+풀 (스무딩 없음)':            compute_perf(v4_values),
}
v4_turnover_comparison = {
    '등가중 벤치마크':                     turnover_stats(ew_turnovers),
    'v1 라우터+풀 (스무딩 없음)':            turnover_stats(hrl_turnovers),
    f'v1 + 스무딩 α={BEST_ALPHA}':          turnover_stats(v1_smoothed_turnovers),
    'v4 라우터+풀 (스무딩 없음)':            turnover_stats(v4_turnovers),
}

v4_perf_df = pd.DataFrame(v4_comparison).T
print()
print(v4_perf_df.to_string())
v4_perf_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_performance.csv', encoding='utf-8-sig')

v4_turnover_df = pd.DataFrame(v4_turnover_comparison).T
print()
print('--- [v4] 턴오버/거래비용 진단 ---')
print(v4_turnover_df.to_string())
v4_turnover_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_turnover_diag.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_performance.csv, crypto_hrl_v4_turnover_diag.csv')

### 13-F. v4에 관성 스무딩 적용 (§9와 동일 α 스윕, step_with_weights로 top-K 재변환 없이 블렌딩)

In [ ]:
%%time
def run_v4_smoothed(alpha):
    """v4 라우터+풀을 테스트 구간에서 실행하되 최종 비중에 관성 스무딩 적용.
    top-K 변환은 비가역적(0이 된 종목의 log를 취할 수 없음)이라 §9의 log-트릭 대신
    action_to_weights()로 얻은 목표 비중을 직접 블렌딩해 step_with_weights()로 반영한다."""
    env = MacroRouterEnvV4(
        buckets=test_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
    )
    obs, _ = env.reset()
    inner = env.inner
    gross, g = [1.0], 1.0
    done = False
    while not done:
        r_action, _ = router_v4.predict(obs, deterministic=True)
        model = env.pool_models[int(r_action)]
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            target_w, rho, top_idx = inner.action_to_weights(a)
            w = (1 - alpha) * inner.weights + alpha * target_w
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            g *= 1 + info['port_ret']
            gross.append(g)
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return np.array(inner.value_history), np.array(gross), list(inner.turnover_history), inner


v4_smooth_results, v4_smooth_curves = {}, {}
_v4_net_by_alpha, _v4_env_by_alpha = {}, {}
for alpha in SMOOTH_ALPHAS:
    label = 'v4 α=1.0 (스무딩 없음)' if alpha == 1.0 else f'v4 α={alpha}'
    print(f'{label} 실행 중...')
    net, gross, turns, inner_env = run_v4_smoothed(alpha)
    v4_smooth_results[label] = smooth_perf_row(net, gross, turns)
    v4_smooth_curves[label]  = net
    _v4_net_by_alpha[alpha]  = net[-1]
    _v4_env_by_alpha[alpha]  = inner_env

BEST_ALPHA_V4 = max((a for a in _v4_net_by_alpha if a < 1.0), key=lambda a: _v4_net_by_alpha[a])

v4_smooth_df = pd.DataFrame(v4_smooth_results).T
print()
print(v4_smooth_df.to_string())
print(f"\n[v4] 비용 후 최고 α = {BEST_ALPHA_V4} (총 수익률 {(_v4_net_by_alpha[BEST_ALPHA_V4]-1)*100:.2f}%)")
v4_smooth_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_smoothing_sweep.csv', encoding='utf-8-sig')
print('저장 완료: crypto_hrl_v4_smoothing_sweep.csv')

fig, ax = plt.subplots(figsize=(14, 5))
for label, curve in v4_smooth_curves.items():
    ax.plot(curve, label=label, lw=1.6)
ax.plot(ew_values, label='등가중 벤치마크', lw=1.5, ls='--', color='black')
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_title(f'[v4] 관성 스무딩 α별 누적 가치 (테스트: {RL_TEST_START} ~, 비용 후)', fontsize=13)
ax.set_ylabel('포트폴리오 가치 (초기=1.0)')
ax.set_xlabel('30분 스텝')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hrl_v4_smoothing_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

### 13-F-보조. 저턴오버 아티팩트 진단 — α=BEST_ALPHA_V4가 정적 포지션인지 확인

In [ ]:
# 13-F 보조 진단 — v4 α=BEST_ALPHA_V4 결과(비용 후 8.26%)가 "초반에 우연히 잡은 포지션이
# 그대로 버틴" 정적 아티팩트인지, 아니면 꾸준한 리밸런싱으로 만들어진 성과인지 확인한다.
# (v3 §12에서 "pullback 단독 +8.05%"가 회전 거의 없는 관측치라 전략으로 인용 금지했던 것과
# 같은 함정이 v4 α=0.01(턴오버 0.06%/스텝)에도 있는지 점검.)

if '_v4_env_by_alpha' in globals() and BEST_ALPHA_V4 in _v4_env_by_alpha:
    diag_env = _v4_env_by_alpha[BEST_ALPHA_V4]
else:
    print(f'캐시된 env 없음 — α={BEST_ALPHA_V4} 재실행...')
    _, _, _, diag_env = run_v4_smoothed(BEST_ALPHA_V4)

turns  = np.array(diag_env.turnover_history)
values = np.array(diag_env.value_history)
n = len(turns)
decile = max(n // 10, 1)

print(f'=== [v4] α={BEST_ALPHA_V4} 정적 포지션 아티팩트 진단 (n={n} 스텝) ===\n')
print(f'턴오버 — 전체 평균 {turns.mean()*100:.4f}% | 앞 10% 평균 {turns[:decile].mean()*100:.4f}% | '
      f'뒤 10% 평균 {turns[-decile:].mean()*100:.4f}% | 뒤 10% 구간 중 턴오버=0인 스텝 비율 '
      f'{(turns[-decile:] == 0).mean()*100:.1f}%')

half = n // 2
first_half_ret  = values[half] / values[0] - 1
second_half_ret = values[-1]  / values[half] - 1
print(f'\n전반부 수익률: {first_half_ret*100:.2f}% | 후반부 수익률: {second_half_ret*100:.2f}%')
print('(후반부가 0%에 가깝거나 음수인데 전반부만 크게 플러스면, 초반에 잡은 포지션이 그대로 버틴 정적 결과라는 뜻)')

# 목표 선택(top-K)이 시간에 따라 실제로 회전하는지 — 앞/뒤 1/4 구간 자산 겹침(자카드 유사도)
sel = diag_env.selection_history
q = max(n // 4, 1)
first_q_assets = set(a for s in sel[:q] for a in s)
last_q_assets  = set(a for s in sel[-q:] for a in s)
jaccard = len(first_q_assets & last_q_assets) / len(first_q_assets | last_q_assets)
print(f'\n앞 1/4 구간 선택 종목 집합: {len(first_q_assets)}개 | 뒤 1/4 구간: {len(last_q_assets)}개 | '
      f'자카드 유사도: {jaccard:.2f} (1.0=완전 동일 종목만 계속 선택, 낮을수록 실제로 회전)')

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(pd.Series(turns).rolling(decile, min_periods=1).mean() * 100, lw=1.3)
ax.set_title(f'[v4] α={BEST_ALPHA_V4} 턴오버 롤링평균(윈도우={decile}스텝) — 시간에 따라 죽어가는지 확인', fontsize=12)
ax.set_ylabel('턴오버 %/스텝')
ax.set_xlabel('30분 스텝')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hrl_v4_turnover_rolling.png', dpi=150, bbox_inches='tight')
plt.show()


### 13-I. episode_progress 학습·서빙 불일치 해소 — "매칭" 라우터 재학습 (한계 3 검증)

In [ ]:
# [v4] 매칭 라우터 재실행용 로더 (재학습 셀을 건너뛴 세션 대비 — idempotent)
if 'router_v4_matched' not in globals():
    router_v4_matched = DQN.load(str(MODEL_DIR / 'crypto_hrl_router_v4_matched.zip'))
    print('[v4] 매칭 라우터 로드 완료')


### 13-I-B. 원본 라우터 vs 매칭 라우터 — 비용 후 성능 비교 (스무딩 α 스윕 포함)

In [ ]:
%%time
def run_v4_router_smoothed(router_model, alpha, episode_progress_mode='real'):
    """§13-F run_v4_smoothed를 일반화 — 라우터 객체와 episode_progress_mode를 인자로 받아
    원본 라우터(router_v4, mode='real')와 매칭 라우터(router_v4_matched, mode='constant')를
    동일 함수로 비교한다."""
    env = MacroRouterEnvV4(
        buckets=test_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
        episode_progress_mode=episode_progress_mode, **env_kwargs,
    )
    obs, _ = env.reset()
    inner = env.inner
    gross, g = [1.0], 1.0
    done = False
    while not done:
        r_action, _ = router_model.predict(obs, deterministic=True)
        model = env.pool_models[int(r_action)]
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            target_w, rho, top_idx = inner.action_to_weights(a)
            w = (1 - alpha) * inner.weights + alpha * target_w
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            g *= 1 + info['port_ret']
            gross.append(g)
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return np.array(inner.value_history), np.array(gross), list(inner.turnover_history)


def _alpha_label(prefix, a):
    return f'{prefix} α=1.0 (스무딩 없음)' if a == 1.0 else f'{prefix} α={a}'


matched_results, _matched_net_by_alpha = {}, {}
for alpha in SMOOTH_ALPHAS:
    label = _alpha_label('v4-matched', alpha)
    print(f'{label} 실행 중...')
    net, gross, turns = run_v4_router_smoothed(router_v4_matched, alpha, episode_progress_mode='constant')
    matched_results[label]      = smooth_perf_row(net, gross, turns)
    _matched_net_by_alpha[alpha] = net[-1]

BEST_ALPHA_V4_MATCHED = max((a for a in _matched_net_by_alpha if a < 1.0), key=lambda a: _matched_net_by_alpha[a])

comparison = {
    '등가중 벤치마크':                                      smooth_results['등가중 벤치마크'],
    _alpha_label('v4 원본 라우터', BEST_ALPHA_V4):          v4_smooth_results[_alpha_label('v4', BEST_ALPHA_V4)],
    _alpha_label('v4 매칭 라우터', BEST_ALPHA_V4_MATCHED):  matched_results[_alpha_label('v4-matched', BEST_ALPHA_V4_MATCHED)],
}
comparison_df = pd.DataFrame(comparison).T
print()
print(comparison_df.to_string())
print(f"\n[매칭 라우터] 비용 후 최고 α = {BEST_ALPHA_V4_MATCHED} (총 수익률 {(_matched_net_by_alpha[BEST_ALPHA_V4_MATCHED]-1)*100:.2f}%)")
comparison_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_matched_comparison.csv', encoding='utf-8-sig')
print('저장 완료: crypto_hrl_v4_matched_comparison.csv')


## 14. §13-K — valid 기반 재검증: α 선택 편향 제거 + 라우터 β 분포 재확인 (2026-07-28)

In [ ]:
# §13-K-A — valid 구간 정의 (v3 §12와 동일 분할 재사용: train 1~9월 / valid 10~12월)
VALID_START, VALID_END = '2025-10-01', '2025-12-31'
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
print(f'valid 구간: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')


### 14-C. α를 valid에서 선택 → test는 그 α로 "1회만" 실행 (선택 편향 제거)

In [ ]:
def run_v4_router_smoothed_on(router_model, alpha, buckets, episode_progress_mode='real'):
    """§13-I-B run_v4_router_smoothed을 buckets 인자로 일반화 — valid/test 어디서든 재사용."""
    env = MacroRouterEnvV4(
        buckets=buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
        episode_progress_mode=episode_progress_mode, **env_kwargs,
    )
    obs, _ = env.reset()
    inner = env.inner
    gross, g = [1.0], 1.0
    done = False
    while not done:
        r_action, _ = router_model.predict(obs, deterministic=True)
        model = env.pool_models[int(r_action)]
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            target_w, rho, top_idx = inner.action_to_weights(a)
            w = (1 - alpha) * inner.weights + alpha * target_w
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            g *= 1 + info['port_ret']
            gross.append(g)
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return np.array(inner.value_history), np.array(gross), list(inner.turnover_history)


valid_alpha_net = {}
for alpha in SMOOTH_ALPHAS:
    label = _alpha_label('v4-matched(valid)', alpha)
    print(f'{label} 실행 중...')
    net, gross, turns = run_v4_router_smoothed_on(router_v4_matched, alpha, valid_buckets, episode_progress_mode='constant')
    valid_alpha_net[alpha] = net[-1]

BEST_ALPHA_V4_VALID = max((a for a in valid_alpha_net if a < 1.0), key=lambda a: valid_alpha_net[a])
print(f"\n[valid에서 선택된] α = {BEST_ALPHA_V4_VALID} (valid 총 수익률 {(valid_alpha_net[BEST_ALPHA_V4_VALID]-1)*100:.2f}%)")


In [ ]:
print(f'[최종 검증] valid에서 고른 α={BEST_ALPHA_V4_VALID}로 test를 "1회만" 실행 (편향 없는 아웃오브샘플 성능 추정치)')
net_clean, gross_clean, turns_clean = run_v4_router_smoothed_on(
    router_v4_matched, BEST_ALPHA_V4_VALID, test_buckets, episode_progress_mode='constant'
)
clean_perf = smooth_perf_row(net_clean, gross_clean, turns_clean)

comparison_clean = {
    '등가중 벤치마크':                                                     smooth_results['등가중 벤치마크'],
    _alpha_label('v4 매칭 (test에서 α선택, 편향 있음)', BEST_ALPHA_V4_MATCHED): matched_results[_alpha_label('v4-matched', BEST_ALPHA_V4_MATCHED)],
    _alpha_label('v4 매칭 (valid에서 α선택, 편향 제거)', BEST_ALPHA_V4_VALID):  clean_perf,
}
comparison_clean_df = pd.DataFrame(comparison_clean).T
print()
print(comparison_clean_df.to_string())
comparison_clean_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_matched_valid_selected.csv', encoding='utf-8-sig')
print('저장 완료: crypto_hrl_v4_matched_valid_selected.csv')


## 15. §14-E — 라우터가 정말 상수함수인가? β=30 단독 실행과 직접 대조 (2026-07-29)

In [ ]:
def run_v4_solo_smoothed_on(beta, alpha, buckets):
    """라우터를 거치지 않고 pool_models_v4[beta] 하나만 계속 사용 — §14-B에서 매칭 라우터가
    β=30에 100% 수렴한 것이 실제로 '상수함수'인지 직접 대조하기 위한 베이스라인.
    run_v4_router_smoothed_on과 동일한 매 스텝 스무딩 로직(step_with_weights)을 그대로 사용."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
    model = pool_models_v4[beta]
    obs, _ = env.reset()
    gross, g = [1.0], 1.0
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
        g *= 1 + info['port_ret']
        gross.append(g)
    return np.array(env.value_history), np.array(gross), list(env.turnover_history)


solo_valid_net, solo_valid_gross, solo_valid_turns = run_v4_solo_smoothed_on(30, BEST_ALPHA_V4_VALID, valid_buckets)
solo_test_net,  solo_test_gross,  solo_test_turns  = run_v4_solo_smoothed_on(30, BEST_ALPHA_V4_MATCHED, test_buckets)

solo_valid_ret = solo_valid_net[-1] / solo_valid_net[0] - 1
solo_test_ret  = solo_test_net[-1]  / solo_test_net[0]  - 1
router_valid_ret = valid_alpha_net[BEST_ALPHA_V4_VALID] - 1
router_test_ret  = _matched_net_by_alpha[BEST_ALPHA_V4_MATCHED] - 1

print('=== [대조] β=30 단독(라우터 없음) vs 매칭 라우터 ===\n')
print(f'valid — β=30 단독: {solo_valid_ret*100:.4f}%  | 매칭 라우터: {router_valid_ret*100:.4f}%  | '
      f'일치: {np.isclose(solo_valid_ret, router_valid_ret, atol=1e-6)}')
print(f'test  — β=30 단독: {solo_test_ret*100:.4f}%   | 매칭 라우터: {router_test_ret*100:.4f}%   | '
      f'일치: {np.isclose(solo_test_ret, router_test_ret, atol=1e-6)}')

solo_test_perf = smooth_perf_row(solo_test_net, solo_test_gross, solo_test_turns)
compare_solo_router = {
    '등가중 벤치마크':                 smooth_results['등가중 벤치마크'],
    'β=30 단독 (라우터 없음)':          solo_test_perf,
    _alpha_label('v4 매칭 라우터', BEST_ALPHA_V4_MATCHED): matched_results[_alpha_label('v4-matched', BEST_ALPHA_V4_MATCHED)],
}
compare_solo_router_df = pd.DataFrame(compare_solo_router).T
print()
print(compare_solo_router_df.to_string())
compare_solo_router_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_solo_vs_router.csv', encoding='utf-8-sig')
print('저장 완료: crypto_hrl_v4_solo_vs_router.csv')


## 16. §15-C — 4개 β 전부 단독 비교: "test에서만 좋은 것"이 β=30만의 문제인가, 레짐 차이인가? (2026-07-29)

In [ ]:
print('=== [§16] 4개 β 에이전트 전부 단독 실행 — valid vs test (라우터 없이, 동일 α=0.01) ===\n')

ALPHA_DIAG = BEST_ALPHA_V4_MATCHED  # 0.01, 지금까지 파이프라인 전체에서 쓰인 값 재사용

ew_valid_values, ew_valid_turnovers = equal_weight_backtest(CryptoPortfolioEnv(valid_buckets, **env_kwargs))

rows_valid, rows_test = {}, {}
rows_valid['등가중 벤치마크'] = smooth_perf_row(ew_valid_values, ew_valid_values, ew_valid_turnovers)
rows_test['등가중 벤치마크']  = smooth_results['등가중 벤치마크']

for beta in POOL_BETAS:
    print(f'β={beta} 실행 중 (valid, test)...')
    v_net, v_gross, v_turns = run_v4_solo_smoothed_on(beta, ALPHA_DIAG, valid_buckets)
    t_net, t_gross, t_turns = run_v4_solo_smoothed_on(beta, ALPHA_DIAG, test_buckets)
    rows_valid[f'β={beta} 단독'] = smooth_perf_row(v_net, v_gross, v_turns)
    rows_test[f'β={beta} 단독']  = smooth_perf_row(t_net, t_gross, t_turns)

valid_df = pd.DataFrame(rows_valid).T
test_df  = pd.DataFrame(rows_test).T
print('\n--- valid(2025 Q4) ---')
print(valid_df.to_string())
print('\n--- test(2026~) ---')
print(test_df.to_string())

valid_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_beta_solo_valid.csv', encoding='utf-8-sig')
test_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_beta_solo_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_beta_solo_valid.csv, crypto_hrl_v4_beta_solo_test.csv')


## 18. TC(거래비용) 민감도 스윕 — 배포 정책이 비용 가정에 얼마나 민감한가 (2026-08-05)

In [ ]:
# §18-A — TC 민감도 스윕 (β=-30 배포 모델, α=0.01 고정, 재학습 없음 — env의 tc 파라미터만 변경)
TC_SWEEP = [0.0005, 0.001, 0.0015, 0.002]  # 5bp / 10bp(배포 기준) / 15bp / 20bp
ALPHA_DEPLOY = 0.01


def run_v4_solo_smoothed_tc(beta, alpha, buckets, tc):
    """run_v4_solo_smoothed_on(§15, 셀80)과 동일하나 env_kwargs의 tc만 오버라이드한다."""
    tc_env_kwargs = {**env_kwargs, 'tc': tc}
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **tc_env_kwargs)
    model = pool_models_v4[beta]
    obs, _ = env.reset()
    gross, g = [1.0], 1.0
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
        g *= 1 + info['port_ret']
        gross.append(g)
    return np.array(env.value_history), np.array(gross), list(env.turnover_history)


def perf_row_tc(net, gross, turnovers, tc):
    """smooth_perf_row(§9, 셀25)와 동일하되, '누적 비용 드래그'를 전역 TC 상수가 아니라
    실제로 이 백테스트에 쓰인 tc로 정확히 계산한다(smooth_perf_row는 셀25 정의 시점의 전역
    TC=0.001을 하드코딩하고 있어 tc를 바꾸는 이 스윕에는 그대로 재사용할 수 없다)."""
    rets   = np.diff(net) / net[:-1]
    ann    = net[-1] ** (48 * 365 / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(48 * 365)
    mdd    = ((net / np.maximum.accumulate(net)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    drag   = 1 - net[-1] / gross[-1]
    return {
        '총 수익률(비용 후)': f'{(net[-1]-1)*100:.2f}%',
        '총 수익률(비용 전)': f'{(gross[-1]-1)*100:.2f}%',
        '연환산':           f'{ann*100:.2f}%',
        'Sharpe':          f'{sharpe:.3f}',
        'MDD':             f'{mdd*100:.2f}%',
        '평균 턴오버/스텝':    f'{t.mean()*100:.2f}%',
        '누적 비용 드래그':    f'{drag*100:.2f}%',
    }


print('=== [§18] TC 민감도 스윕 — β=-30, α=0.01, valid/test ===\n')
rows_tc_valid, rows_tc_test = {}, {}
for tc in TC_SWEEP:
    label = f'TC={tc*10000:.0f}bp' + (' (배포 기준)' if np.isclose(tc, TC) else '')
    print(f'{label} 실행 중...')
    v_net, v_gross, v_turns = run_v4_solo_smoothed_tc(-30, ALPHA_DEPLOY, valid_buckets, tc)
    t_net, t_gross, t_turns = run_v4_solo_smoothed_tc(-30, ALPHA_DEPLOY, test_buckets, tc)
    rows_tc_valid[label] = perf_row_tc(v_net, v_gross, v_turns, tc)
    rows_tc_test[label]  = perf_row_tc(t_net, t_gross, t_turns, tc)

tc_valid_df = pd.DataFrame(rows_tc_valid).T
tc_test_df  = pd.DataFrame(rows_tc_test).T
print('\n--- TC별 valid(2025 Q4) ---')
print(tc_valid_df.to_string())
print('\n--- TC별 test(2026~) ---')
print(tc_test_df.to_string())

tc_valid_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_tc_sweep_valid.csv', encoding='utf-8-sig')
tc_test_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_tc_sweep_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_tc_sweep_{valid,test}.csv')